# The Hard Hexagon model

![logo](hexagon.svg)

Tensor networks are a natural way to do statistical mechanics on a lattice.
As an example of this we will extract the central charge of the hard hexagon model.
This model is known to have central charge $0.8$, and has very peculiar non-local (anyonic) symmetries.
Because TensorKit supports anyonic symmetries, so does MPSKit.
To follow the tutorial you need the following packages.

In [ ]:
using MPSKit, TensorKit, Plots, Polynomials
# Model constructors for the MPSKit documentation and examples.
# Adapted from MPSKitModels.jl v0.4.7, src/models/{hamiltonians,transfermatrices}.jl:
# https://github.com/QuantumKitHub/MPSKitModels.jl/tree/v0.4.7/src/models
# Operators come from TensorKitTensors; chain assembly uses MPSKit directly.
#
# MIT License
# Copyright (c) 2021 Maarten Van Damme
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

module ExampleModels

using MPSKit, TensorKit
using TensorKitTensors: SpinOperators, BosonOperators, HubbardOperators

export chain_hamiltonian, transverse_field_ising, heisenberg_XXX
export hubbard_model, bose_hubbard_model, hard_hexagon

# L specifies an open finite chain; otherwise unitcell specifies an infinite chain.
function chain_hamiltonian(twosite, onesite = nothing; L = nothing, unitcell::Integer = 1)
    nsites = isnothing(L) ? unitcell : L
    nsites isa Integer && nsites > 0 || throw(ArgumentError("chain length must be a positive integer"))
    nbonds = isnothing(L) ? nsites : nsites - 1
    terms = Pair[(i, i + 1) => twosite for i in 1:nbonds]
    if !isnothing(onesite)
        append!(terms, [i => onesite for i in 1:nsites])
    end
    spaces = fill(space(twosite, 1), nsites)
    constructor = isnothing(L) ? InfiniteMPOHamiltonian : FiniteMPOHamiltonian
    return constructor(spaces, terms)
end

# H = -J sum(σz_i σz_{i+1} + g σx_i).
function transverse_field_ising(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, g = 1.0, kwargs...
    )
    ZZ = 4 * SpinOperators.S_z_S_z(T, symmetry; spin = 1 // 2)
    X = SpinOperators.σˣ(T, symmetry)
    return chain_hamiltonian(-J * ZZ, -J * g * X; kwargs...)
end
transverse_field_ising(symmetry::Type{<:Sector}; kwargs...) =
    transverse_field_ising(ComplexF64, symmetry; kwargs...)

# H = J sum(S_i ⋅ S_{i+1}); preserve the examples' spin-1 default.
function heisenberg_XXX(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, spin = 1, kwargs...
    )
    term = J * SpinOperators.S_exchange(T, symmetry; spin)
    return chain_hamiltonian(term; kwargs...)
end
heisenberg_XXX(symmetry::Type{<:Sector}; kwargs...) =
    heisenberg_XXX(ComplexF64, symmetry; kwargs...)

# H = -t sum(e⁺_i e⁻_{i+1} + h.c.) + U sum(n↑ n↓) - mu sum(n).
function hubbard_model(
        T::Type{<:Number} = ComplexF64, particle_symmetry::Type{<:Sector} = Trivial,
        spin_symmetry::Type{<:Sector} = Trivial;
        t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    # e_hopping includes the fermionic reordering sign and is Hermitian.
    hopping = HubbardOperators.e_hopping(T, particle_symmetry, spin_symmetry)
    interaction = HubbardOperators.ud_num(T, particle_symmetry, spin_symmetry)
    N = HubbardOperators.e_num(T, particle_symmetry, spin_symmetry)
    return chain_hamiltonian(-t * hopping, U * interaction - mu * N; kwargs...)
end

# H = -t sum(b⁺_i b⁻_{i+1} + h.c.) + U/2 sum(n(n-1)) - mu sum(n).
function bose_hubbard_model(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        cutoff::Integer = 5, t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    hopping = BosonOperators.b_plus_b_min(T, symmetry; cutoff) +
        BosonOperators.b_min_b_plus(T, symmetry; cutoff)
    N = BosonOperators.b_num(T, symmetry; cutoff)
    interaction = N * (N - id(domain(N)))
    return chain_hamiltonian(-t * hopping, U / 2 * interaction - mu * N; kwargs...)
end

function hard_hexagon(T::Type{<:Number} = ComplexF64)
    P = Vect[FibonacciAnyon](:τ => 1)
    O = ones(T, P ⊗ P ← P ⊗ P)
    block(O, FibonacciAnyon(:I)) .*= 0
    return InfiniteMPO([O])
end

end

using .ExampleModels

The [hard hexagon model](https://en.wikipedia.org/wiki/Hard_hexagon_model) is a 2-dimensional lattice model of a gas, where particles are allowed to be on the vertices of a triangular lattice, but no two particles may be adjacent.
This can be encoded in a transfer matrix with a local MPO tensor using anyonic symmetries, and the resulting MPO is defined in our shared [model helpers](https://github.com/QuantumKitHub/MPSKit.jl/blob/main/docs/src/assets/models.jl).

In order to use these anyonic symmetries, we need to generalise the notion of the bond dimension and define how it interacts with the symmetry.
Thus, we implement a way of converting integers to symmetric spaces of the given dimension, which provides a crude guess for how the final MPS would distribute its Schmidt spectrum.

In [ ]:
mpo = hard_hexagon()
P = physicalspace(mpo, 1)
function virtual_space(D::Integer)
    _D = round(Int, D / sum(dim, values(FibonacciAnyon)))
    return Vect[FibonacciAnyon](sector => _D for sector in (:I, :τ))
end

@assert isapprox(dim(virtual_space(100)), 100; atol = 3)

## The leading boundary

One way to study statistical mechanics in infinite systems with tensor networks is by approximating the dominant eigenvector of the transfer matrix by an MPS.
This dominant eigenvector contains a lot of hidden information.
For example, the free energy density can be extracted by computing the dominant eigenvalue of the transfer matrix.
Additionally, we can compute the entanglement entropy as well as the correlation length of the state:

In [ ]:
D = 10
V = virtual_space(D)
ψ₀ = InfiniteMPS([P], [V])
ψ, envs, info = leading_boundary(
    ψ₀, mpo,
    VUMPS(; verbosity = 0, alg_eigsolve = MPSKit.Defaults.alg_eigsolve(; ishermitian = false))
) # use non-hermitian eigensolver
F = -log(real(leading_eigenvalue(ψ, mpo)))
S = real(first(entropy(ψ)))
ξ = correlation_length(ψ; sector = leftunit(ψ))
println("F = $F\tS = $S\tξ = $ξ")

## The scaling hypothesis

The dominant eigenvector is of course only an approximation.
The finite bond dimension enforces a finite correlation length, which effectively introduces a length scale in the system.
This can be exploited to formulate a scaling hypothesis [pollmann2009](@cite), which in turn allows to extract the central charge.

First we need to know the entropy and correlation length at several different bond dimensions.
Our approach will be to re-use the previous approximated dominant eigenvector, and then expanding its bond dimension and re-running VUMPS.
According to the scaling hypothesis we should have $S ∝ \frac{c}{6} log(ξ)$. Therefore we should find $c$ using

In [ ]:
function scaling_simulations(
        ψ₀, mpo, Ds; verbosity = 0, tol = 1.0e-6,
        alg_eigsolve = MPSKit.Defaults.alg_eigsolve(; ishermitian = false)
    )
    entropies = similar(Ds, Float64)
    correlations = similar(Ds, Float64)
    alg = VUMPS(; verbosity, tol, alg_eigsolve)
    sector = unit(sectortype(mpo)) # in this example the dominant correlation functions are in the trivial sector

    ψ, envs, = leading_boundary(ψ₀, mpo, alg)
    entropies[1] = real(entropy(ψ)[1])
    correlations[1] = correlation_length(ψ; sector)

    for (i, d) in enumerate(diff(Ds))
        ψ, envs = changebonds(ψ, mpo, OptimalExpand(; trunc = truncrank(d)), envs)
        ψ, envs, = leading_boundary(ψ, mpo, alg, envs)
        entropies[i + 1] = real(entropy(ψ)[1])
        correlations[i + 1] = correlation_length(ψ; sector)
    end
    return entropies, correlations
end

bond_dimensions = 10:5:25
ψ₀ = InfiniteMPS([P], [virtual_space(bond_dimensions[1])])
Ss, ξs = scaling_simulations(ψ₀, mpo, bond_dimensions)

f = fit(log.(ξs), 6 * Ss, 1)
c = f.coeffs[2]
p = plot(; xlabel = "logarithmic correlation length", ylabel = "entanglement entropy")
p = plot(log.(ξs), Ss; seriestype = :scatter, label = nothing)
plot!(p, ξ -> f(ξ) / 6; label = "fit")

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*